In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#import library

import numpy as np
import random

#PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models
import torchvision.transforms.functional as F
from torchvision.transforms import v2
from torch.utils.data import DataLoader, random_split
from torchvision.io import read_image
from torchvision.utils import make_grid
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

#Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Metrics
from sklearn.metrics import confusion_matrix

#Image
from PIL import Image, ImageOps, ImageEnhance

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [5]:
#data augumentation
#create transformer
imgSize = 112

#train transformer
train_transformer = v2.Compose([
    v2.RandomRotation(degrees = 20),
    v2.RandomHorizontalFlip(p = 0.3),
    v2.RandomVerticalFlip(p = 0.3),
    v2.Resize(size = (imgSize, imgSize)),
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    v2.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

#Validation transformer
validation_transformer = v2.Compose([
    v2.Resize(size = (imgSize, imgSize)),
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    v2.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [8]:
#load model
model = torch.load("/content/drive/MyDrive/melanoma/model.pth", map_location=torch.device('cpu'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/melanoma/model.pth'

In [ ]:
#define function to show image
def show(imgs):
    if not isinstance(imgs, list):
        imgs = [imgs]
    fig, axs = plt.subplots(ncols=len(imgs), squeeze=False)
    for i, img in enumerate(imgs):
        img = img.detach()
        img = F.to_pil_image(img)
        axs[0, i].imshow(np.asarray(img))
        axs[0, i].set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])

In [ ]:
#specify test image directory
benign_test_path = "/content/drive/MyDrive/melanoma/dataset/test/Benign/"
malignant_test_path = "/content/drive/MyDrive/melanoma/dataset/test/Malignant/"

if random.randint(1,10) % 2 == 0:
  #choose image in malinant
  img_path = malignant_test_path + str(random.randint(5602, 6602)) + ".jpg"
  print("ACTUAL VALUE: this is MALIGNANT image")
else:
  #choose image in benign
  img_path = benign_test_path + str(random.randint(6299, 7300)) + ".jpg"
  print("ACTUAL VALUE: this is BENIGN image")

img = read_image(img_path)
show(img)

#preprocess image
input = validation_transformer(img)

#add batch dimension
input = input.unsqueeze(0)

#feed forward
output = model(input)

print(f'{output.item():.2f}')

if output.item() < 0 :
  print("model predcition: BENIGN")
else:
  print("model predcition: MALIGNANT")